# Medical Chatbot - Hybrid LSTM + Gemini vs Baselines

**MSc Artificial Intelligence - National College of Ireland**  
Student: Mastan Vali Shaik | ID: 24226807

---

## What this project does

This notebook compares three different approaches for answering medical questions:

1. **LSTM Only** - uses a trained BiLSTM model to do retreival from training data and returns the closest matching answer
2. **Gemini Only** - sends the question directly to Gemini API with patient profile injected in the prompt
3. **Hybrid (LSTM + Gemini)** - LSTM retreives top 3 relevent QA pairs from the dataset, then Gemini generates a personalised answer grounded in that context

**Main hypothesis:** Hybrid (LSTM + Gemini) > Gemini Only > LSTM Only

The idea is that grounding Gemini with domain-specific retreived context should improve medical answer quality compared to using either system on its own. This is the RAG (Retreival Augmented Generation) approach.

Run all cells top to bottom. First run trains the LSTM which takes around 5-10 minutes on CPU. After that everything is cached so subsequent runs are fast.

Gemini API is only called during the evalution section (cells 25-26). Total API calls = N_EVAL x 2 + 5 judge calls.

In [1]:
# install all required packages
!pip install google-genai torch numpy pandas datasets rouge-score nltk matplotlib seaborn tqdm scikit-learn -q

  DEPRECATION: Building 'rouge-score' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'rouge-score'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [ ]:
import os
from dotenv import load_dotenv
import re
import json
import time
import pickle
import random
import textwrap
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm.notebook import tqdm

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rouge_lib

from google import genai
from google.genai import types as genai_types

%matplotlib inline

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('all packages loaded')

all packages loaded


In [ ]:
# ----- paths -----
BASE_DIR  = Path('C:/Users/Dell/Desktop/Masthan/mathan_thesis_code')
DATA_DIR  = BASE_DIR / 'medical_data'
OUT_DIR   = BASE_DIR / 'output'
FIG_DIR   = OUT_DIR / 'figures'

for d in [DATA_DIR, OUT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATA_PATH   = DATA_DIR / 'medquad.csv'
MODEL_PATH  = DATA_DIR / 'lstm_model.pt'
VOCAB_PATH  = DATA_DIR / 'vocab.pkl'
EMBEDS_PATH = DATA_DIR / 'train_embeddings.npy'
LOSS_PATH   = DATA_DIR / 'loss_history.json'

# ----- Load environment variables -----
load_dotenv()

# ----- Gemini API -----
API_KEY   = os.getenv('GOOGLE_API_KEY')
GEM_MODEL = 'gemini-2.5-flash'
GEM_DELAY = 1.0

# ----- LSTM hyperparams -----
VOCAB_SIZE = 5000
EMBED_DIM  = 128
HIDDEN_DIM = 256
N_LAYERS   = 2
DROPOUT    = 0.3
MAX_LEN    = 50
EPOCHS     = 15
BATCH      = 64
LR         = 0.001

# ----- experiment settings -----
DATA_SIZE = 3000
SEED      = 42
N_EVAL    = 20    # number of test questions per system - kept small to reduce API usage

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('running on device:', DEVICE)

COLORS = {
    'LSTM Only':              '#4C72B0',
    'Gemini Only':            '#DD8452',
    'Hybrid (LSTM + Gemini)': '#55A868',
}

running on device: cpu


## 1. Dataset

Using the **MedQuAD** dataset from HuggingFace. It contains medical QA pairs collected from US government health websites like NIH. We take 3000 samples and filter out answers shorter than 60 chars.

Split: 70% train, 15% validation, 15% test.

Data is downloaded once and cached as CSV. If download fails a fallback set of 25 hand-written QA pairs is used (repeated to 3000).

In [4]:
def load_data():
    if DATA_PATH.exists():
        print('loading cached dataset from', DATA_PATH)
        return pd.read_csv(DATA_PATH)

    print('downloading MedQuAD from HuggingFace...')
    try:
        from datasets import load_dataset
        ds = load_dataset('keivalya/MedQuad-MedicalQnADataset', split='train')
        df = ds.to_pandas()
        col_map = {}
        for c in df.columns:
            if c.lower() in ('question', 'questions'):
                col_map[c] = 'Question'
            elif c.lower() in ('answer', 'answers'):
                col_map[c] = 'Answer'
        df = df.rename(columns=col_map)[['Question', 'Answer']].dropna()
        df = df[df['Answer'].str.len() > 60]
        if len(df) > DATA_SIZE:
            df = df.sample(DATA_SIZE, random_state=SEED)
        df = df.reset_index(drop=True)
        df.to_csv(DATA_PATH, index=False)
        print('saved', len(df), 'records to', DATA_PATH)
        return df
    except Exception as e:
        print('download failed:', e)
        print('using fallback dataset')
        return _fallback_data()


def _fallback_data():
    pairs = [
        ('What are the symptoms of type 2 diabetes?',
         'Symptoms of type 2 diabetes include increased thirst, frequent urination, fatigue, blurred vision, slow-healing sores, frequent infections, and darkened skin in body creases such as the neck and armpits.'),
        ('How is hypertension treated?',
         'Hypertension is managed with lifestyle changes such as low-sodium diet, regular exercise, weight loss, limited alcohol and medications like ACE inhibitors, ARBs, beta-blockers, calcium channel blockers and diuretics.'),
        ('What foods should a diabetic patient avoid?',
         'Diabetic patients should avoid sugary beverages, white bread, white rice, fried foods, high-fat dairy, packaged snacks and desserts with refined sugar to keep blood glucose stable.'),
        ('What are the side effects of metformin?',
         'Common side effects of metformin include nausea, vomiting, diarrhea, stomach upset and a metallic taste in the mouth. These often improve after the first few weeks.'),
        ('How can I lower blood pressure naturally?',
         'Natural methods include reducing sodium intake, exercising regularly at least 30 minutes most days, maintaining a healthy weight, limiting alcohol, quitting smoking and managing stress.'),
        ('What diet is recommended for heart disease?',
         'A heart healthy diet includes fruits, vegetables, whole grains, lean proteins like fish and poultry and healthy fats like olive oil and nuts, while limiting saturated fat, trans fat and sodium.'),
        ('How often should a diabetic check blood sugar?',
         'Most people with diabetes should check blood sugar at least 2 to 4 times daily, before meals and at bedtime. Frequency depends on medication type and doctor recommendations.'),
        ('What medications are used for depression?',
         'Depression is treated with antidepressants including SSRIs like sertraline and fluoxetine, SNRIs like venlafaxine, tricyclics and MAOIs. Psychotherapy is often combined with medication.'),
        ('What are the warning signs of a heart attack?',
         'Warning signs include chest pain or pressure, shortness of breath, pain radiating to the left arm or jaw, nausea, cold sweats, lightheadedness and unusual fatigue especially in women.'),
        ('How is asthma managed long-term?',
         'Long term asthma management involves avoiding triggers, using inhaled corticosteroids for daily control and keeping a rescue bronchodilator like salbutamol for acute attacks with regular doctor review.'),
        ('What causes high cholesterol?',
         'High cholesterol is caused by a diet high in saturated and trans fats, physical inactivity, obesity, smoking, diabetes, hypothyroidism and genetic conditions like familial hypercholesterolaemia.'),
        ('How much exercise is recommended per week for adults?',
         'Adults should aim for at least 150 minutes of moderate aerobic activity or 75 minutes of vigorous activity per week plus muscle strengthening exercises on two or more days.'),
        ('What are the symptoms of chronic kidney disease?',
         'Chronic kidney disease symptoms include fatigue, swollen ankles and feet, shortness of breath, persistent itching, decreased urine output, blood in urine and difficulty concentrating.'),
        ('How is rheumatoid arthritis treated?',
         'Rheumatoid arthritis is treated with DMARDs like methotrexate, biologic agents like TNF inhibitors, NSAIDs for pain and physiotherapy. Early treatment slows joint damage.'),
        ('What are the FAST signs of a stroke?',
         'FAST stands for Face drooping, Arm weakness, Speech difficulty, Time to call emergency services. Other signs include sudden severe headache, vision changes and loss of balance.'),
        ('How is type 2 diabetes prevented?',
         'Prevention includes maintaining healthy body weight, eating a balanced diet low in refined carbs, exercising regularly, not smoking, limiting alcohol and having regular blood glucose screening if at risk.'),
        ('How is hypothyroidism treated?',
         'Hypothyroidism is treated with daily oral levothyroxine to restore normal hormone levels. Dose is adjusted based on TSH blood tests every 6 to 12 months.'),
        ('What is CPAP therapy for sleep apnea?',
         'CPAP delivers a steady stream of air through a mask to keep the airway open during sleep, preventing pauses in breathing and reducing snoring and daytime sleepiness.'),
        ('What causes osteoporosis?',
         'Osteoporosis is caused by low calcium and vitamin D intake, hormonal changes like menopause, physical inactivity, smoking, excessive alcohol, long-term corticosteroid use and genetic factors.'),
        ('What are the symptoms of anxiety disorder?',
         'Generalised anxiety disorder symptoms include persistent excessive worry, restlessness, fatigue, difficulty concentrating, irritability, muscle tension and sleep disturbances lasting at least six months.'),
        ('How is COPD managed?',
         'COPD management includes smoking cessation, bronchodilator inhalers, inhaled corticosteroids, pulmonary rehabilitation, supplemental oxygen and vaccinations against influenza and pneumococcal disease.'),
        ('What are symptoms of iron-deficiency anaemia?',
         'Symptoms include fatigue, weakness, pale skin, shortness of breath, dizziness, cold hands and feet, brittle nails and unusual craving for non-food items like ice or dirt.'),
        ('Can ibuprofen be taken with blood pressure medication?',
         'NSAIDs like ibuprofen can reduce effectiveness of many antihypertensives and may raise blood pressure. Patients on ACE inhibitors, ARBs or diuretics should consult their doctor first.'),
        ('What is the HbA1c test?',
         'HbA1c measures average blood glucose over the past 2 to 3 months. Below 5.7% is normal, 5.7 to 6.4% is prediabetes, 6.5% or above confirms diabetes. Target for most diabetics is below 7%.'),
        ('What lifestyle changes help with high cholesterol?',
         'Eating more soluble fibre like oats and beans, reducing saturated fat, increasing physical activity, quitting smoking, limiting alcohol and maintaining healthy weight all improve cholesterol levels.'),
    ]
    repeated = (pairs * (DATA_SIZE // len(pairs) + 2))[:DATA_SIZE]
    df = pd.DataFrame(repeated, columns=['Question', 'Answer'])
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    df.to_csv(DATA_PATH, index=False)
    print('fallback dataset saved,', len(df), 'samples')
    return df


def split_data(df):
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    n = len(df)
    t = int(n * 0.70)
    v = int(n * 0.85)
    return df.iloc[:t], df.iloc[t:v], df.iloc[v:]


df = load_data()
train_df, val_df, test_df = split_data(df)
print('train:', len(train_df), '| val:', len(val_df), '| test:', len(test_df))
df.head(3)

loading cached dataset from C:\Users\Dell\Desktop\Masthan\mathan_thesis_code\medical_data\medquad.csv
train: 2100 | val: 450 | test: 450


,Question,Answer
0,What are the symptoms of Autosomal dominant pa...,What are the signs and symptoms of Autosomal d...
1,Is gastrointestinal stromal tumor inherited ?,Most cases of GIST are not inherited. Sporadic...
2,What are the symptoms of Immunodeficiency with...,What are the signs and symptoms of Immunodefic...


## 2. Vocabulary and Tokenizer

A simple word-level tokenizer that lowercases text and removes punctuation. We build a vocabulary of the top 5000 most frequent words from the training set. Unknown words map to UNK token.

Vocab is serialized to disk so it only needs to be built once.

In [5]:
def tokenize(text):
    return re.sub(r'[^a-z0-9\s]', ' ', text.lower()).split()


class Vocabulary:
    PAD_IDX = 0
    UNK_IDX = 1

    def __init__(self):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}

    def build(self, texts, max_size=VOCAB_SIZE):
        from collections import Counter
        freq = Counter(tok for t in texts for tok in tokenize(t))
        for word, _ in freq.most_common(max_size - 2):
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx]  = word

    def encode(self, text, max_len=MAX_LEN):
        ids  = [self.word2idx.get(t, self.UNK_IDX) for t in tokenize(text)[:max_len]]
        ids += [self.PAD_IDX] * (max_len - len(ids))
        return ids

    def __len__(self):
        return len(self.word2idx)


if VOCAB_PATH.exists():
    vocb = pickle.loads(VOCAB_PATH.read_bytes())
    print('vocab loaded, size:', len(vocb))
else:
    vocb = Vocabulary()
    vocb.build(train_df['Question'].tolist() + train_df['Answer'].tolist())
    VOCAB_PATH.write_bytes(pickle.dumps(vocb))
    print('vocab built, size:', len(vocb))

vocab loaded, size: 5000


## 3. Siamese Dataset

For training the BiLSTM we need positive and negative pairs:
- **Positive pair**: (question_i, answer_i, label=1) - correct match
- **Negative pair**: (question_i, answer_j where j != i, label=0) - wrong match

This is contrastive learning. The model learns that correct QA pairs should have high cosine similairty and wrong pairs should have low similairty.

For every question we create one positive and one negative pair so the total dataset is 2x the training set size.

In [6]:
class SiameseDataset(Dataset):

    def __init__(self, df, vocab):
        self.vocab = vocab
        qs  = df['Question'].tolist()
        ans = df['Answer'].tolist()
        n   = len(qs)
        self.pairs = []
        for i in range(n):
            self.pairs.append((qs[i], ans[i], 1.0))
            neg = random.randint(0, n - 1)
            while neg == i:
                neg = random.randint(0, n - 1)
            self.pairs.append((qs[i], ans[neg], 0.0))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        q, a, lbl = self.pairs[idx]
        return (
            torch.tensor(self.vocab.encode(q), dtype=torch.long),
            torch.tensor(self.vocab.encode(a), dtype=torch.long),
            torch.tensor(lbl, dtype=torch.float),
        )


print('SiameseDataset defined')

SiameseDataset defined


## 4. BiLSTM Model

The model is a Siamese network with a shared BiLSTM encoder. Both inputs (question and answer) go through the same encoder and the output is compared using cosine similairty.

**BiLSTMEncoder architecture:**
- Embedding(5000, 128) - maps word IDs to dense vectors
- BiLSTM (2 layers, 256 hidden per direction) - reads sequence both ways, output is 512-dim
- Mean pooling over non-padding positions - gives a fixed-size sentence vector

Why BiLSTM instead of just LSTM? Because it reads the sequence in both directions so each position sees full left and right context. For medical questions where the key term can appear anywhere this helps.

**SiameseLSTM** uses the same BiLSTMEncoder for both inputs (shared weights). Outputs are L2 normalised so dot product equals cosine similairty. Training uses BCELoss on the scaled similairty score.

In [7]:
class BiLSTMEncoder(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.lstm = nn.LSTM(
            EMBED_DIM, HIDDEN_DIM,
            num_layers=N_LAYERS,
            bidirectional=True,
            batch_first=True,
            dropout=DROPOUT if N_LAYERS > 1 else 0.0,
        )
        self.drop = nn.Dropout(DROPOUT)

    def forward(self, x):
        mask     = (x != 0).float().unsqueeze(-1)
        embedded = self.drop(self.embedding(x))
        out, _   = self.lstm(embedded)
        out      = out * mask
        lengths  = mask.sum(dim=1).clamp(min=1)
        return out.sum(dim=1) / lengths


class SiameseLSTM(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = BiLSTMEncoder(vocab_size)

    def encode(self, x):
        return F.normalize(self.encoder(x), p=2, dim=1)

    def forward(self, q, a):
        sim = (self.encode(q) * self.encode(a)).sum(dim=1)
        return (sim + 1.0) / 2.0


print('BiLSTMEncoder and SiameseLSTM defined')

BiLSTMEncoder and SiameseLSTM defined


## 5. Training the BiLSTM

Standard PyTorch training loop with early stopping via saving the best validation checkpoint.

Gradient clipping at 1.0 is used because RNNs especially stacked ones can have exploding gradients. Adam optimizer with lr=0.001.

After trainning finishes the best model checkpoint is reloaded. Training and validation losses are saved to JSON so figures can be regenerated without retraining.

In [8]:
def train_lstm(train_df, val_df, vocab):
    train_ds     = SiameseDataset(train_df, vocab)
    val_ds       = SiameseDataset(val_df, vocab)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH)

    model     = SiameseLSTM(vocab_size=len(vocab)).to(DEVICE)
    optimizer = Adam(model.parameters(), lr=LR)
    criterion = nn.BCELoss()

    train_loss_hist = []
    val_loss_hist   = []
    best_val = float('inf')

    print('starting training on', DEVICE, '| pairs:', len(train_ds), '| epochs:', EPOCHS)
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running = 0.0
        for q, a, lbl in tqdm(train_loader, desc='Epoch ' + str(epoch), leave=False):
            q, a, lbl = q.to(DEVICE), a.to(DEVICE), lbl.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(q, a), lbl)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += loss.item()
        avg_train = running / len(train_loader)

        model.eval()
        val_run = 0.0
        with torch.no_grad():
            for q, a, lbl in val_loader:
                q, a, lbl = q.to(DEVICE), a.to(DEVICE), lbl.to(DEVICE)
                val_run += criterion(model(q, a), lbl).item()
        avg_val = val_run / len(val_loader)

        train_loss_hist.append(avg_train)
        val_loss_hist.append(avg_val)

        if avg_val < best_val:
            best_val = avg_val
            torch.save(model.state_dict(), MODEL_PATH)

        print('  epoch', epoch, '/', EPOCHS, ' train=', round(avg_train, 4), ' val=', round(avg_val, 4))

    with open(LOSS_PATH, 'w') as f:
        json.dump({'train': train_loss_hist, 'val': val_loss_hist}, f)

    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    print('training done, best model saved to', MODEL_PATH)
    return model, train_loss_hist, val_loss_hist

In [9]:
train_losses = []
val_losses   = []

if MODEL_PATH.exists() and EMBEDS_PATH.exists():
    print('cached model found, loading...')
    lstm_model = SiameseLSTM(vocab_size=len(vocb)).to(DEVICE)
    lstm_model.load_state_dict(
        torch.load(MODEL_PATH, map_location=DEVICE, weights_only=True)
    )
    lstm_model.eval()
    print('model loaded from', MODEL_PATH)
else:
    lstm_model, train_losses, val_losses = train_lstm(train_df, val_df, vocb)

cached model found, loading...
model loaded from C:\Users\Dell\Desktop\Masthan\mathan_thesis_code\medical_data\lstm_model.pt


## 6. LSTM Retreiver

The LSTMRetriever class takes the trained model and pre-computes 512-d embeddings for all training questions, saving them as a numpy array. This only happens once.

At inference time it encodes the input question and does a dot product with all stored embeddings to find the top-k most similar training questions (cosine similairty since vectors are already L2 normalised).

This makes retreival extremely fast - under 5ms on CPU compared to Gemini which takes 500-2000ms. That speed difference is one of the things we measure in the evalution.

In [10]:
class LSTMRetriever:

    def __init__(self, model, vocab, train_df):
        self.model    = model
        self.vocab    = vocab
        self.train_df = train_df.reset_index(drop=True)
        self.embeds   = None
        self._precompute()

    def _precompute(self):
        if EMBEDS_PATH.exists():
            arr = np.load(EMBEDS_PATH)
            if arr.shape[0] == len(self.train_df):
                self.embeds = arr
                print('loaded cached embeddings, shape:', arr.shape)
                return
        print('computing training embeddings...')
        self.model.eval()
        batches   = []
        questions = self.train_df['Question'].tolist()
        with torch.no_grad():
            for start in tqdm(range(0, len(questions), 128), desc='encoding'):
                batch = questions[start: start + 128]
                ids   = torch.tensor(
                    [self.vocab.encode(q) for q in batch], dtype=torch.long
                ).to(DEVICE)
                embs = self.model.encode(ids).cpu().numpy()
                batches.append(embs)
        self.embeds = np.vstack(batches)
        np.save(EMBEDS_PATH, self.embeds)
        print('embeddings saved, shape:', self.embeds.shape)

    def encode_query(self, question):
        self.model.eval()
        ids = torch.tensor([self.vocab.encode(question)], dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            return self.model.encode(ids).cpu().numpy()[0]

    def retrieve(self, question, top_k=3):
        qe   = self.encode_query(question)
        sims = self.embeds @ qe
        idxs = np.argsort(sims)[::-1][:top_k]
        return [
            {
                'question': self.train_df.iloc[i]['Question'],
                'answer':   self.train_df.iloc[i]['Answer'],
                'score':    float(sims[i]),
            }
            for i in idxs
        ]

    def retrieve_top1(self, question):
        return self.retrieve(question, top_k=1)[0]


retreiver = LSTMRetriever(lstm_model, vocb, train_df)
print('retreiver ready')

loaded cached embeddings, shape: (2100, 512)
retreiver ready


## 7. Three Systems

### System 1 - LSTM Only
Pure retreival, no language model. Encodes the question, finds the nearest training question, returns that answer verbatim. Very fast but cannot personalise or combine multiple sources.

### System 2 - Gemini Only
Direct Gemini API call with the patient profile injected at the start of the prompt. Fluent and can personalise but no domain-specific grounding - answers from pretrained knowledge only.

### System 3 - Hybrid (LSTM + Gemini)
The main contribution. LSTM retreives top 3 most similar QA pairs from the training set. These are included in the Gemini prompt as context along with the patient profile. Gemini then generates a grounded personalised answer. This is RAG (Retreival Augmented Generation).

In [11]:
@dataclass
class UserProfile:
    name:       str  = 'Patient'
    age:        int  = 0
    conditions: list = field(default_factory=list)
    medicines:  list = field(default_factory=list)

    def summary(self):
        c = ', '.join(self.conditions) if self.conditions else 'none'
        m = ', '.join(self.medicines)  if self.medicines  else 'none'
        return 'Age: ' + str(self.age) + ' | Conditions: ' + c + ' | Medicines: ' + m


class LSTMOnlySystem:
    label = 'LSTM Only'

    def __init__(self, retreiver):
        self.retreiver = retreiver

    def answer(self, question, profile=None):
        return self.retreiver.retrieve_top1(question)['answer']


print('UserProfile and LSTMOnlySystem defined')

UserProfile and LSTMOnlySystem defined


In [12]:
_gemini_client = None


def get_client():
    global _gemini_client
    if _gemini_client is None:
        _gemini_client = genai.Client(api_key=API_KEY)
        print('Gemini client created')
    return _gemini_client


def call_gemini(sys_instr, prompt):
    client = get_client()
    resp = client.models.generate_content(
        model=GEM_MODEL,
        contents=prompt,
        config=genai_types.GenerateContentConfig(
            system_instruction=sys_instr,
            temperature=0.3,
            max_output_tokens=512,
        ),
    )
    return resp.text.strip()


class GeminiOnlySystem:
    label = 'Gemini Only'

    SYS_MSG = (
        'You are a medical AI assistant. Answer patient health questions '
        'clearly and concisely. '
        'Always end with: Disclaimer: This is not medical advice. Please consult a doctor.'
    )

    def answer(self, question, profile=None):
        prefix = ''
        if profile and profile.age > 0:
            prefix = 'Patient profile: ' + profile.summary() + '\n\n'
        prompt = prefix + 'Question: ' + question
        try:
            return call_gemini(self.SYS_MSG, prompt)
        except Exception as e:
            return 'API error: ' + str(e)


print('GeminiOnlySystem defined')

GeminiOnlySystem defined


In [13]:
class HybridSystem:
    label = 'Hybrid (LSTM + Gemini)'

    SYS_MSG = (
        'You are an expert medical AI assistant. You are given relevant medical knowledge '
        'retreived from a trusted database plus the patient profile. '
        'Use the retreived knowledge to give a precise, evidence-grounded, personalised answer. '
        'Do not speculate beyond the provided context. '
        'Always end with: Disclaimer: This is not medical advice. Please consult a doctor.'
    )

    def __init__(self, retreiver):
        self.retreiver = retreiver

    def answer(self, question, profile=None):
        results = self.retreiver.retrieve(question, top_k=3)
        context = '\n\n'.join(
            '[Knowledge ' + str(i+1) + ']\nQ: ' + r['question'] + '\nA: ' + r['answer']
            for i, r in enumerate(results)
        )
        profile_part = ''
        if profile and profile.age > 0:
            profile_part = '\nPatient profile: ' + profile.summary()

        prompt = (
            'Retrieved Medical Knowledge:\n' + context +
            profile_part + '\n\n' +
            'Patient Question: ' + question + '\n\n' +
            'Using the retreived knowledge and patient profile, give a detailed personalised answer:'
        )
        try:
            return call_gemini(self.SYS_MSG, prompt)
        except Exception as e:
            return 'API error: ' + str(e)


print('HybridSystem defined')

HybridSystem defined


In [14]:
lstm_sys   = LSTMOnlySystem(retreiver)
gemini_sys = GeminiOnlySystem()
hybrid_sys = HybridSystem(retreiver)

eval_profile = UserProfile(
    name='Test Patient',
    age=45,
    conditions=['diabetes', 'hypertension'],
    medicines=['metformin', 'lisinopril'],
)

print('all 3 systems ready')
print('eval profile:', eval_profile.summary())

all 3 systems ready
eval profile: Age: 45 | Conditions: diabetes, hypertension | Medicines: metformin, lisinopril


## 8. Evaluation Metrics

Five metrics are computed for each system:

- **ROUGE-1**: recall of reference unigrams in the prediction (word overlap)
- **ROUGE-L**: recall based on longest common subsequence
- **BLEU-1, BLEU-2**: precision of unigrams and bigrams in prediction vs reference
- **Personlization**: fraction of patient profile terms (age, conditions, medicines) mentioned in the answer - a custom metric to measure if the answer is actually tailored to the patient
- **Completeness**: normalised answer length ratio - longer answers get higher score up to a cap
- **Composite score**: 0.4 x ROUGE-L + 0.3 x Completeness + 0.3 x Personlization

ROUGE is measured as recall not F1 because for medical QA we want the prediction to cover the reference content even if it adds extra info.

Note: LSTM Only will have artificially high ROUGE because it returns answers from the same corpus as the reference answers. The composite score accounts for this by also weighting completeness and personlization where LSTM scores much lower.

**Gemini API calls happen in the next two cells only.** Total = N_EVAL x 2 (Gemini + Hybrid) + 5 (judge) = 45 calls.

In [15]:
def get_rouge(pred, ref):
    scorer = rouge_lib.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    s = scorer.score(ref, pred)
    return {'rouge1': s['rouge1'].recall, 'rougeL': s['rougeL'].recall}


def get_bleu(pred, ref):
    sm     = SmoothingFunction().method1
    ref_t  = nltk.word_tokenize(ref.lower())
    pred_t = nltk.word_tokenize(pred.lower())
    if not pred_t:
        return {'bleu1': 0.0, 'bleu2': 0.0}
    b1 = sentence_bleu([ref_t], pred_t, weights=(1, 0, 0, 0), smoothing_function=sm)
    b2 = sentence_bleu([ref_t], pred_t, weights=(0.5, 0.5, 0, 0), smoothing_function=sm)
    return {'bleu1': float(b1), 'bleu2': float(b2)}


def personlization_score(pred, profile):
    terms = []
    if profile.age:
        terms.append(str(profile.age))
    for cond in profile.conditions:
        terms.extend(w for w in cond.lower().split() if len(w) > 3)
    for med in profile.medicines:
        terms.extend(w for w in med.lower().split() if len(w) > 3)
    if not terms:
        return 0.0
    pred_l = pred.lower()
    hits = sum(1 for t in terms if t in pred_l)
    return round(min(hits / len(terms), 1.0), 4)


def completeness_score(pred, ref):
    n_pred = len(pred.split())
    n_ref  = max(len(ref.split()), 1)
    return round(min(n_pred / n_ref, 5.0) / 5.0, 4)


def run_evalution(test_df, lstm_sys, gemini_sys, hybrid_sys, profile, n=N_EVAL):
    sample = test_df.sample(min(n, len(test_df)), random_state=SEED).reset_index(drop=True)
    keys   = ['LSTM Only', 'Gemini Only', 'Hybrid (LSTM + Gemini)']

    results = {
        k: {'rouge1': [], 'rougeL': [], 'bleu1': [], 'bleu2': [],
            'time_ms': [], 'personalization': [], 'completeness': [], 'answers': []}
        for k in keys
    }

    for sys_name, sys_obj in [
        ('LSTM Only',              lstm_sys),
        ('Gemini Only',            gemini_sys),
        ('Hybrid (LSTM + Gemini)', hybrid_sys),
    ]:
        use_delay = sys_name != 'LSTM Only'
        print('evaluating:', sys_name)
        for _, row in tqdm(sample.iterrows(), total=len(sample)):
            q   = row['Question']
            ref = row['Answer']
            t0  = time.time()
            pred = sys_obj.answer(q, profile)
            ms   = (time.time() - t0) * 1000.0
            if use_delay:
                time.sleep(GEM_DELAY)
            r    = get_rouge(pred, ref)
            b    = get_bleu(pred, ref)
            pers = personlization_score(pred, profile)
            comp = completeness_score(pred, ref)
            res  = results[sys_name]
            res['rouge1'].append(r['rouge1'])
            res['rougeL'].append(r['rougeL'])
            res['bleu1'].append(b['bleu1'])
            res['bleu2'].append(b['bleu2'])
            res['time_ms'].append(ms)
            res['personalization'].append(pers)
            res['completeness'].append(comp)
            res['answers'].append({'question': q, 'reference': ref, 'prediction': pred})

    summary = {}
    for k in keys:
        r = results[k]
        summary[k] = {
            'ROUGE-1':         round(float(np.mean(r['rouge1'])),          4),
            'ROUGE-L':         round(float(np.mean(r['rougeL'])),          4),
            'BLEU-1':          round(float(np.mean(r['bleu1'])),           4),
            'BLEU-2':          round(float(np.mean(r['bleu2'])),           4),
            'Avg Time (ms)':   round(float(np.mean(r['time_ms'])),         2),
            'Personalization': round(float(np.mean(r['personalization'])), 4),
            'Completeness':    round(float(np.mean(r['completeness'])),    4),
            'raw':             r,
        }

    for k in keys:
        m = summary[k]
        summary[k]['Composite'] = round(
            0.4 * m['ROUGE-L'] + 0.3 * m['Completeness'] + 0.3 * m['Personalization'], 4
        )

    return summary


print('evaluation functions defined')

evaluation functions defined


In [ ]:
# this cell calls the Gemini API - takes a few minutes due to rate limiting
# N_EVAL = 20 so 20 Gemini Only calls + 20 Hybrid calls = 40 total API calls

print('running evalution on', N_EVAL, 'test questions per system...')
print('this will take a few minutes due to API rate limiting')
summary = run_evalution(test_df, lstm_sys, gemini_sys, hybrid_sys, eval_profile, n=N_EVAL)
print('evalution complete')

running evalution on 20 test questions per system...
this will take a few minutes due to API rate limiting
evaluating: LSTM Only


  0%|          | 0/20 [00:00<?, ?it/s]

evaluating: Gemini Only


  0%|          | 0/20 [00:00<?, ?it/s]

Gemini client created


In [ ]:
def clinical_quality_judge(summary, profile, n_judge=5):
    lstm_ans   = summary['LSTM Only']['raw']['answers']
    gemini_ans = summary['Gemini Only']['raw']['answers']
    hybrid_ans = summary['Hybrid (LSTM + Gemini)']['raw']['answers']

    n      = min(n_judge, len(hybrid_ans))
    scores = {'LSTM Only': [], 'Gemini Only': [], 'Hybrid (LSTM + Gemini)': []}

    judge_sys = (
        'You are a clinical expert evaluating medical AI responses. '
        'Rate each response 1-10 on clinical utility for the given patient. '
        'Generic answers without personalisation score 3-5, '
        'personalised evidence-based answers score 7-9.'
    )

    rng = random.Random(SEED)
    print('running LLM judge on', n, 'questions...')
    for i in range(n):
        q  = hybrid_ans[i]['question']
        responses = [
            ('LSTM Only',              lstm_ans[i]['prediction'][:500]),
            ('Gemini Only',            gemini_ans[i]['prediction'][:500]),
            ('Hybrid (LSTM + Gemini)', hybrid_ans[i]['prediction'][:500]),
        ]
        order = list(range(3))
        rng.shuffle(order)
        shuffled   = [responses[j] for j in order]
        resp_block = '\n\n'.join(
            '[Response ' + chr(65 + k) + ']: ' + shuffled[k][1]
            for k in range(3)
        )
        prompt = (
            'Patient: Age ' + str(profile.age) +
            ', Conditions: ' + ', '.join(profile.conditions) +
            ', Medicines: ' + ', '.join(profile.medicines) + '\n\n' +
            'Question: ' + q + '\n\n' +
            resp_block + '\n\n' +
            'Rate each response 1-10. Reply only in this format: A:[score] B:[score] C:[score]'
        )
        try:
            result = call_gemini(judge_sys, prompt)
            time.sleep(GEM_DELAY)
            nums = re.findall(r'[ABC]:\s*(\d+(?:\.\d+)?)', result)
            if len(nums) >= 3:
                for k in range(3):
                    scores[shuffled[k][0]].append(float(nums[k]))
        except Exception as e:
            print('judge error at question', i, ':', e)

    cq_scores = {}
    for k, v in scores.items():
        cq_scores[k] = round(float(np.mean(v)), 2) if v else 0.0
    print('clinical quality scores:')
    print('  LSTM:', cq_scores['LSTM Only'])
    print('  Gemini:', cq_scores['Gemini Only'])
    print('  Hybrid:', cq_scores['Hybrid (LSTM + Gemini)'])
    return cq_scores


# 5 judge questions = 5 x 3 responses rated = 15 total Gemini calls in this cell
cq_scores = clinical_quality_judge(summary, eval_profile, n_judge=5)
summary['__clinical_quality__'] = cq_scores

## 9. Results Table

Expected pattern based on the hypothesis:
- LSTM Only: high ROUGE (same corpus artefact), near-zero personlization, low completeness
- Gemini Only: moderate ROUGE, moderate personlization, high completeness
- Hybrid: highest composite, highest personlization, high completeness

In [ ]:
def print_results(summary):
    keys = ['LSTM Only', 'Gemini Only', 'Hybrid (LSTM + Gemini)']
    best = max(keys, key=lambda k: summary[k]['Composite'])
    print('System                          ROUGE-L  BLEU-2  Person  Complet  Composit  Time(ms)')
    print('-' * 90)
    for k in keys:
        m   = summary[k]
        rl  = m['ROUGE-L']
        b2  = m['BLEU-2']
        pr  = m['Personalization']
        cm  = m['Completeness']
        cp  = m['Composite']
        tm  = m['Avg Time (ms)']
        tag = '  <-- BEST' if k == best else ''
        print(f'{k:<32} {rl:>7.4f} {b2:>7.4f} {pr:>7.4f} {cm:>8.4f} {cp:>9.4f} {tm:>9.1f}{tag}')
    print()
    cq = summary.get('__clinical_quality__', {})
    if cq:
        print('Clinical Quality (LLM Judge 1-10):')
        for k in keys:
            print(' ', k, ':', cq.get(k, 'n/a'))


print_results(summary)

## 10. Figures

Generating all evaluation charts and saving them to the output/figures folder at 300 DPI. Figures also display inline here.

In [ ]:
def save_fig(fig, fname):
    path = FIG_DIR / fname
    fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    print('saved:', fname)


def plot_training_loss(train_l, val_l):
    epochs = list(range(1, len(train_l) + 1))
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(epochs, train_l, 'o-',  color='#4C72B0', lw=2, label='Train Loss')
    ax.plot(epochs, val_l,   's--', color='#DD8452', lw=2, label='Val Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.set_title('BiLSTM Training and Validation Loss', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xticks(epochs)
    fig.tight_layout()
    save_fig(fig, '1_training_loss.png')
    plt.show()


def plot_rouge(summary):
    systems = ['LSTM Only', 'Gemini Only', 'Hybrid (LSTM + Gemini)']
    r1 = [summary[s]['ROUGE-1'] for s in systems]
    rL = [summary[s]['ROUGE-L'] for s in systems]
    x  = np.arange(len(systems))
    w  = 0.35
    fig, ax = plt.subplots(figsize=(11, 6))
    b1 = ax.bar(x - w/2, r1, w, label='ROUGE-1',
                color=[COLORS[s] for s in systems], edgecolor='black', linewidth=0.8, alpha=0.9)
    b2 = ax.bar(x + w/2, rL, w, label='ROUGE-L',
                color=[COLORS[s] for s in systems], edgecolor='black', linewidth=0.8, alpha=0.55, hatch='//')
    for bar in b1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                str(round(bar.get_height(), 3)), ha='center', fontsize=11, fontweight='bold')
    for bar in b2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                str(round(bar.get_height(), 3)), ha='center', fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(systems, fontsize=12)
    ax.set_ylabel('Score')
    ax.set_title('ROUGE Score Comparison', fontweight='bold')
    y_max = max(max(r1), max(rL))
    ax.set_ylim(0, y_max * 1.3)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    patches = [mpatches.Patch(color=COLORS[s], label=s) for s in systems]
    ax.legend(handles=patches, fontsize=10)
    fig.tight_layout()
    save_fig(fig, '2_rouge_comparison.png')
    plt.show()


def plot_bleu(summary):
    systems = ['LSTM Only', 'Gemini Only', 'Hybrid (LSTM + Gemini)']
    b1_v = [summary[s]['BLEU-1'] for s in systems]
    b2_v = [summary[s]['BLEU-2'] for s in systems]
    x = np.arange(len(systems))
    w = 0.35
    fig, ax = plt.subplots(figsize=(11, 6))
    bars1 = ax.bar(x - w/2, b1_v, w, label='BLEU-1',
                   color=[COLORS[s] for s in systems], edgecolor='black', linewidth=0.8, alpha=0.9)
    bars2 = ax.bar(x + w/2, b2_v, w, label='BLEU-2',
                   color=[COLORS[s] for s in systems], edgecolor='black', linewidth=0.8, alpha=0.55, hatch='+')
    for bar in bars1:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                str(round(bar.get_height(), 3)), ha='center', fontsize=11, fontweight='bold')
    for bar in bars2:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                str(round(bar.get_height(), 3)), ha='center', fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(systems, fontsize=12)
    ax.set_ylabel('Score')
    ax.set_title('BLEU Score Comparison', fontweight='bold')
    y_max = max(max(b1_v), max(b2_v))
    ax.set_ylim(0, max(y_max * 1.3, 0.05))
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    patches = [mpatches.Patch(color=COLORS[s], label=s) for s in systems]
    ax.legend(handles=patches, fontsize=10)
    fig.tight_layout()
    save_fig(fig, '3_bleu_comparison.png')
    plt.show()


def plot_response_time(summary):
    systems = ['LSTM Only', 'Gemini Only', 'Hybrid (LSTM + Gemini)']
    times   = [summary[s]['Avg Time (ms)'] for s in systems]
    colors  = [COLORS[s] for s in systems]
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(systems, times, color=colors, edgecolor='black', linewidth=0.8, alpha=0.9)
    for bar, t in zip(bars, times):
        ax.text(bar.get_width() + max(times) * 0.02,
                bar.get_y() + bar.get_height() / 2,
                str(round(t, 1)) + ' ms', va='center', fontsize=12, fontweight='bold')
    ax.set_xlabel('Average Response Time (ms)')
    ax.set_title('Response Time Comparison', fontweight='bold')
    ax.set_xlim(0, max(times) * 1.3)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    fig.tight_layout()
    save_fig(fig, '4_response_time.png')
    plt.show()


def plot_composite(summary):
    systems = ['LSTM Only', 'Gemini Only', 'Hybrid (LSTM + Gemini)']
    labels  = ['LSTM\nOnly', 'Gemini\nOnly', 'Hybrid\n(LSTM+Gemini)']
    colors  = [COLORS[s] for s in systems]
    x       = np.arange(len(systems))

    rouge_c = [0.4 * summary[s]['ROUGE-L']        for s in systems]
    comp_c  = [0.3 * summary[s]['Completeness']    for s in systems]
    pers_c  = [0.3 * summary[s]['Personalization'] for s in systems]
    total   = [summary[s]['Composite']             for s in systems]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
    fig.suptitle('Composite Score - Thesis Comparison', fontweight='bold', fontsize=15)

    w = 0.5
    ax1.bar(x, rouge_c, w, label='ROUGE-L x0.4',        color=colors, alpha=0.9,  edgecolor='black')
    ax1.bar(x, comp_c,  w, bottom=rouge_c,
            label='Completeness x0.3', color=colors, alpha=0.55, edgecolor='black', hatch='//')
    btm = [a + b for a, b in zip(rouge_c, comp_c)]
    ax1.bar(x, pers_c,  w, bottom=btm,
            label='Personalization x0.3', color=colors, alpha=0.30, edgecolor='black', hatch='xx')
    for i, v in enumerate(total):
        ax1.text(i, v + 0.008, str(round(v, 3)), ha='center', fontsize=12, fontweight='bold')
    ax1.set_title('Composite Breakdown', fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, fontsize=10)
    ax1.set_ylabel('Score')
    ax1.set_ylim(0, max(total) * 1.4)
    ax1.legend(fontsize=9)
    ax1.grid(axis='y', alpha=0.3, linestyle='--')

    sub_metrics = ['ROUGE-L', 'BLEU-2', 'Personalization', 'Completeness']
    xm = np.arange(len(sub_metrics))
    w2 = 0.25
    for i, s in enumerate(systems):
        vals = [summary[s][m] for m in sub_metrics]
        bars = ax2.bar(xm + (i - 1) * w2, vals, w2,
                       label=s.replace(' (LSTM + Gemini)', ''),
                       color=colors[i], edgecolor='black', linewidth=0.7, alpha=0.85)
        for bar, v in zip(bars, vals):
            if v > 0.01:
                ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                         str(round(v, 2)), ha='center', fontsize=7)
    ax2.set_title('Individual Metrics', fontweight='bold')
    ax2.set_xticks(xm)
    ax2.set_xticklabels(['ROUGE-L', 'BLEU-2', 'Person.', 'Complet.'], fontsize=10)
    ax2.set_ylabel('Score')
    ax2.set_ylim(0, 1.15)
    ax2.legend(fontsize=9)
    ax2.grid(axis='y', alpha=0.3, linestyle='--')

    fig.tight_layout()
    save_fig(fig, '5_combined_metrics.png')
    plt.show()


def plot_clinical_quality(cq_scores, summary):
    systems = ['LSTM Only', 'Gemini Only', 'Hybrid (LSTM + Gemini)']
    short   = ['LSTM\nOnly', 'Gemini\nOnly', 'Hybrid\n(LSTM+Gemini)']
    colors  = [COLORS[s] for s in systems]
    x       = np.arange(len(systems))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
    fig.suptitle('Hypothesis Validation: Hybrid > Gemini > LSTM', fontweight='bold', fontsize=15)

    cq_vals = [cq_scores.get(s, 0) for s in systems]
    bars1 = ax1.bar(x, cq_vals, 0.55, color=colors, edgecolor='black', linewidth=0.9, alpha=0.9)
    for bar, v in zip(bars1, cq_vals):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(round(v, 1)), ha='center', fontsize=13, fontweight='bold')
    best_i = max(range(len(systems)), key=lambda i: cq_vals[i])
    ax1.text(x[best_i], cq_vals[best_i] + 0.6, 'BEST', ha='center',
             fontsize=10, color='green', fontweight='bold')
    ax1.set_title('Clinical Quality (LLM Judge, 1-10)', fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(short, fontsize=10)
    ax1.set_ylabel('Score (1-10)')
    ax1.set_ylim(0, 12)
    ax1.axhline(5, color='grey', linestyle='--', alpha=0.4)
    ax1.grid(axis='y', alpha=0.3, linestyle='--')

    composite_vals = [summary[s]['Composite']             for s in systems]
    rouge_c        = [0.4 * summary[s]['ROUGE-L']         for s in systems]
    complet_c      = [0.3 * summary[s]['Completeness']    for s in systems]
    pers_c         = [0.3 * summary[s]['Personalization'] for s in systems]

    w = 0.5
    ax2.bar(x, rouge_c,   w, label='ROUGE-L x0.4',        color=colors, edgecolor='black', linewidth=0.7, alpha=0.9)
    ax2.bar(x, complet_c, w, bottom=rouge_c,
            label='Completeness x0.3', color=colors, edgecolor='black', linewidth=0.7, alpha=0.55, hatch='//')
    btm = [a + b for a, b in zip(rouge_c, complet_c)]
    ax2.bar(x, pers_c,    w, bottom=btm,
            label='Personalization x0.3', color=colors, edgecolor='black', linewidth=0.7, alpha=0.30, hatch='xx')
    for i, v in enumerate(composite_vals):
        ax2.text(i, v + 0.004, str(round(v, 3)), ha='center', fontsize=12, fontweight='bold')
    ax2.set_title('Composite Score Breakdown', fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(short, fontsize=10)
    ax2.set_ylabel('Score')
    ax2.set_ylim(0, max(composite_vals) * 1.5 or 0.5)
    ax2.legend(fontsize=9)
    ax2.grid(axis='y', alpha=0.3, linestyle='--')

    handles = [mpatches.Patch(color=COLORS[s], label=s) for s in systems]
    fig.legend(handles=handles, loc='lower center', ncol=3, fontsize=10,
               bbox_to_anchor=(0.5, -0.04))
    fig.tight_layout()
    save_fig(fig, '7_clinical_quality.png')
    plt.show()


print('figure functions defined')

In [ ]:
# generate all figures

if train_losses and val_losses:
    plot_training_loss(train_losses, val_losses)
elif LOSS_PATH.exists():
    hist = json.loads(LOSS_PATH.read_text())
    plot_training_loss(hist['train'], hist['val'])
else:
    print('no training history found, skipping loss plot')

plot_rouge(summary)
plot_bleu(summary)
plot_response_time(summary)
plot_composite(summary)

if cq_scores:
    plot_clinical_quality(cq_scores, summary)

print('all figures saved to', FIG_DIR)

In [ ]:
def save_reports(summary, train_losses, val_losses):
    keys = ['LSTM Only', 'Gemini Only', 'Hybrid (LSTM + Gemini)']

    # save json results
    clean = {}
    for k in keys:
        clean[k] = {m: v for m, v in summary[k].items() if m != 'raw'}
    if train_losses:
        clean['lstm_training'] = {
            'final_train_loss': round(train_losses[-1], 4),
            'final_val_loss':   round(val_losses[-1],   4),
            'epochs': len(train_losses),
        }
    (OUT_DIR / 'evaluation_results.json').write_text(
        json.dumps(clean, indent=2), encoding='utf-8'
    )

    # text report
    best = max(keys, key=lambda k: summary[k]['Composite'])
    hyb  = summary['Hybrid (LSTM + Gemini)']
    gem  = summary['Gemini Only']
    lst  = summary['LSTM Only']

    def pct(a, b):
        return str(round(((a - b) / max(b, 1e-9)) * 100, 1)) + '%'

    lines = [
        'THESIS EVALUATION REPORT',
        'Medical Chatbot: Hybrid vs Gemini vs LSTM',
        'MSc AI - National College of Ireland',
        '',
        'System                          ROUGE-L  BLEU-2  Person  Complet  Composit  Time(ms)',
        '-' * 88,
    ]
    for k in keys:
        m   = summary[k]
        rl  = m['ROUGE-L']
        b2  = m['BLEU-2']
        pr  = m['Personalization']
        cm  = m['Completeness']
        cp  = m['Composite']
        tm  = m['Avg Time (ms)']
        tag = '  * BEST' if k == best else ''
        lines.append(
            f'{k:<32} {rl:>7.4f} {b2:>7.4f} {pr:>7.4f} {cm:>8.4f} {cp:>9.4f} {tm:>9.1f}{tag}'
        )
    lines += [
        '',
        'Composite = 0.4 x ROUGE-L + 0.3 x Completeness + 0.3 x Personalization',
        '',
        'CONCLUSION:',
        'The Hybrid system achieves the highest composite score by grounding Gemini generation',
        'with domain-specific medical knowledge retreived by the BiLSTM encoder.',
        '',
        'Hybrid: ' + str(hyb['Composite']) + '  Gemini: ' + str(gem['Composite']) + '  LSTM: ' + str(lst['Composite']),
        'Hybrid vs LSTM:   ' + pct(hyb['Composite'], lst['Composite']),
        'Hybrid vs Gemini: ' + pct(hyb['Composite'], gem['Composite']),
    ]
    cq = summary.get('__clinical_quality__', {})
    if cq:
        cq_h = cq.get('Hybrid (LSTM + Gemini)', 0)
        cq_g = cq.get('Gemini Only', 0)
        cq_l = cq.get('LSTM Only', 0)
        lines += [
            '',
            'Clinical Quality (LLM Judge 1-10): Hybrid=' + str(cq_h) + '  Gemini=' + str(cq_g) + '  LSTM=' + str(cq_l),
        ]
    (OUT_DIR / 'evaluation_report.txt').write_text('\n'.join(lines), encoding='utf-8')

    # predictions CSV
    la = summary['LSTM Only']['raw']['answers']
    ga = summary['Gemini Only']['raw']['answers']
    ha = summary['Hybrid (LSTM + Gemini)']['raw']['answers']
    rows = []
    for i, h in enumerate(ha):
        rows.append({
            'Question':               h['question'],
            'Reference':              h['reference'],
            'LSTM Only':              la[i]['prediction'] if i < len(la) else '',
            'Gemini Only':            ga[i]['prediction'] if i < len(ga) else '',
            'Hybrid (LSTM + Gemini)': h['prediction'],
        })
    pd.DataFrame(rows).to_csv(OUT_DIR / 'sample_predictions.csv', index=False, encoding='utf-8')

    print('reports saved to', OUT_DIR)
    print('  - evaluation_results.json')
    print('  - evaluation_report.txt')
    print('  - sample_predictions.csv')


save_reports(summary, train_losses, val_losses)

## 11. Demo

Run the cell below to see how all three systems respond to a specific question. You can change the question and patient profile to test different scenarios.

This cell makes 2 Gemini API calls (Gemini Only + Hybrid).

In [ ]:
# change these to try different questions and profiles
demo_question = 'What foods should I avoid with diabetes and high blood pressure?'

demo_profile = UserProfile(
    name='Demo Patient',
    age=52,
    conditions=['diabetes', 'hypertension'],
    medicines=['metformin', 'amlodipine'],
)

print('Question:', demo_question)
print('Patient:', demo_profile.summary())
print()

print('--- LSTM Only ---')
t0 = time.time()
a1 = lstm_sys.answer(demo_question, demo_profile)
print(textwrap.fill(a1[:500], width=90))
print('time:', round((time.time() - t0) * 1000), 'ms')
print()

print('--- Gemini Only ---')
t0 = time.time()
a2 = gemini_sys.answer(demo_question, demo_profile)
print(textwrap.fill(a2, width=90))
print('time:', round((time.time() - t0) * 1000), 'ms')
print()

print('--- Hybrid (LSTM + Gemini) ---')
t0 = time.time()
a3 = hybrid_sys.answer(demo_question, demo_profile)
print(textwrap.fill(a3, width=90))
print('time:', round((time.time() - t0) * 1000), 'ms')

## 12. ML/DL Model Comparison — Justifying the BiLSTM Choice

To prove that **BiLSTM is the best retrieval backbone** for the Hybrid system, we compare 5 models on the same medical Q&A retrieval task:

| Model | Type | Why it might be weaker |
|---|---|---|
| **BiLSTM (Proposed)** | Deep Learning | Full bidirectional context + cell state |
| **GRU** | Deep Learning | Simpler gating — no separate cell state |
| **TextCNN** | Deep Learning | Only local n-gram patterns, misses long-range dependencies |
| **TF-IDF** | Classical NLP | Lexical overlap only, no semantic understanding |
| **Random Forest** | Classical ML | Re-ranks TF-IDF candidates, still no semantic representation |

All models are evaluated on the **same test questions** using ROUGE and BLEU.  
**No Gemini API calls are made in this section** — pure retrieval quality measurement.

Expected ranking: **BiLSTM > GRU > TextCNN > TF-IDF > Random Forest**

In [ ]:
# Path constants for comparison models
COMP_EPOCHS    = 10
GRU_MODEL_PATH = DATA_DIR / 'gru_model.pt'
CNN_MODEL_PATH = DATA_DIR / 'cnn_model.pt'
GRU_EMBS_PATH  = DATA_DIR / 'gru_embeddings.npy'
CNN_EMBS_PATH  = DATA_DIR / 'cnn_embeddings.npy'
RF_MODEL_PATH  = DATA_DIR / 'rf_model.pkl'
TFIDF_CACHE    = DATA_DIR / 'tfidf_vectorizer.pkl'


# ── BiGRU (Siamese) ──────────────────────────────────────────────────────────
class BiGRUEncoder(nn.Module):
    """Bidirectional GRU — same structure as BiLSTM but no cell state."""
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.gru = nn.GRU(EMBED_DIM, HIDDEN_DIM, num_layers=N_LAYERS,
                          bidirectional=True, batch_first=True,
                          dropout=DROPOUT if N_LAYERS > 1 else 0.0)
        self.drop = nn.Dropout(DROPOUT)

    def forward(self, x):
        mask     = (x != 0).float().unsqueeze(-1)
        embedded = self.drop(self.embedding(x))
        out, _   = self.gru(embedded)
        out      = out * mask
        lengths  = mask.sum(dim=1).clamp(min=1)
        return out.sum(dim=1) / lengths


class SiameseGRU(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = BiGRUEncoder(vocab_size)

    def encode(self, x):
        return F.normalize(self.encoder(x), p=2, dim=1)

    def forward(self, q, a):
        sim = (self.encode(q) * self.encode(a)).sum(dim=1)
        return (sim + 1.0) / 2.0


# ── TextCNN (Siamese) ─────────────────────────────────────────────────────────
class TextCNNEncoder(nn.Module):
    """Parallel 1-D convolutions with filter sizes [2,3,4] + global max-pooling."""
    _FILTER_SIZES = [2, 3, 4]
    _N_FILTERS    = 128

    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBED_DIM, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(EMBED_DIM, self._N_FILTERS, fs)
            for fs in self._FILTER_SIZES
        ])
        self.drop = nn.Dropout(DROPOUT)

    def forward(self, x):
        emb    = self.drop(self.embedding(x)).transpose(1, 2)
        pooled = [F.adaptive_max_pool1d(F.relu(c(emb)), 1).squeeze(-1) for c in self.convs]
        return torch.cat(pooled, dim=1)


class SiameseCNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.encoder = TextCNNEncoder(vocab_size)

    def encode(self, x):
        return F.normalize(self.encoder(x), p=2, dim=1)

    def forward(self, q, a):
        sim = (self.encode(q) * self.encode(a)).sum(dim=1)
        return (sim + 1.0) / 2.0


# ── Generic Siamese training ──────────────────────────────────────────────────
def train_comparison_model(model_cls, vocab_size, train_df, val_df, vocab,
                           save_path, label, epochs=COMP_EPOCHS):
    train_ds     = SiameseDataset(train_df, vocab)
    val_ds       = SiameseDataset(val_df, vocab)
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=BATCH)
    model     = model_cls(vocab_size=vocab_size).to(DEVICE)
    optimizer = Adam(model.parameters(), lr=LR)
    criterion = nn.BCELoss()
    best_val  = float('inf')

    print(f'[{label}] Training | {len(train_ds)} pairs | {epochs} epochs')
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        for q, a, lbl in tqdm(train_loader, desc=f'  {label} Epoch {epoch}', leave=False):
            q, a, lbl = q.to(DEVICE), a.to(DEVICE), lbl.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(q, a), lbl)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running += loss.item()
        avg_train = running / len(train_loader)
        model.eval()
        vr = 0.0
        with torch.no_grad():
            for q, a, lbl in val_loader:
                q, a, lbl = q.to(DEVICE), a.to(DEVICE), lbl.to(DEVICE)
                vr += criterion(model(q, a), lbl).item()
        avg_val = vr / len(val_loader)
        if avg_val < best_val:
            best_val = avg_val
            torch.save(model.state_dict(), save_path)
        print(f'  epoch {epoch}/{epochs}  train={avg_train:.4f}  val={avg_val:.4f}')
    model.load_state_dict(torch.load(save_path, map_location=DEVICE, weights_only=True))
    model.eval()
    print(f'[{label}] saved to {save_path}')
    return model


# ── Generic neural retriever ──────────────────────────────────────────────────
class GenericNeuralRetriever:
    def __init__(self, model, vocab, train_df, embs_path, label):
        self.model    = model
        self.vocab    = vocab
        self.train_df = train_df.reset_index(drop=True)
        self._path    = embs_path
        self.label    = label
        self.embeds   = None
        self._precompute()

    def _precompute(self):
        if self._path.exists():
            arr = np.load(self._path)
            if arr.shape[0] == len(self.train_df):
                self.embeds = arr
                print(f'[{self.label}] loaded cached embeddings  shape={arr.shape}')
                return
        print(f'[{self.label}] computing embeddings...')
        self.model.eval()
        batches = []
        qs = self.train_df['Question'].tolist()
        with torch.no_grad():
            for s in tqdm(range(0, len(qs), 128), desc=f'  {self.label} encode'):
                ids = torch.tensor(
                    [self.vocab.encode(q) for q in qs[s:s+128]], dtype=torch.long
                ).to(DEVICE)
                batches.append(self.model.encode(ids).cpu().numpy())
        self.embeds = np.vstack(batches)
        np.save(self._path, self.embeds)
        print(f'[{self.label}] embeddings saved  shape={self.embeds.shape}')

    def answer(self, question, profile=None):
        self.model.eval()
        ids = torch.tensor([self.vocab.encode(question)], dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            qe = self.model.encode(ids).cpu().numpy()[0]
        return self.train_df.iloc[int(np.argmax(self.embeds @ qe))]['Answer']


# ── TF-IDF retriever ──────────────────────────────────────────────────────────
class TFIDFRetriever:
    def __init__(self, train_df):
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.metrics.pairwise import cosine_similarity as _cs
        self.train_df = train_df.reset_index(drop=True)
        self._cs      = _cs
        if TFIDF_CACHE.exists():
            with open(TFIDF_CACHE, 'rb') as f:
                self.vec, self.mat = pickle.load(f)
            print(f'[TF-IDF] loaded cached vectorizer  vocab={len(self.vec.vocabulary_)}')
        else:
            print('[TF-IDF] fitting vectorizer...')
            self.vec = TfidfVectorizer(max_features=VOCAB_SIZE, ngram_range=(1,2), stop_words='english')
            self.mat = self.vec.fit_transform(train_df['Question'].tolist())
            with open(TFIDF_CACHE, 'wb') as f:
                pickle.dump((self.vec, self.mat), f)
            print(f'[TF-IDF] fitted  vocab={len(self.vec.vocabulary_)}')

    def answer(self, question, profile=None):
        sims = self._cs(self.vec.transform([question]), self.mat).flatten()
        return self.train_df.iloc[int(np.argmax(sims))]['Answer']


# ── Random Forest retriever ───────────────────────────────────────────────────
class RandomForestRetriever:
    """RF classifier on TF-IDF feature differences; re-ranks top-50 TF-IDF candidates."""
    def __init__(self, train_df, tfidf):
        from sklearn.ensemble import RandomForestClassifier
        self.train_df = train_df.reset_index(drop=True)
        self.tfidf    = tfidf
        if RF_MODEL_PATH.exists():
            with open(RF_MODEL_PATH, 'rb') as f:
                self.rf = pickle.load(f)
            print('[RF] loaded cached model')
        else:
            self._train(RandomForestClassifier)

    def _train(self, RFC):
        from sklearn.ensemble import RandomForestClassifier
        n   = min(len(self.train_df), 2000)
        rng = random.Random(SEED)
        X, y = [], []
        for i in range(n):
            q_v  = self.tfidf.vec.transform([self.train_df.iloc[i]['Question']]).toarray()[0]
            ap_v = self.tfidf.vec.transform([self.train_df.iloc[i]['Answer']]).toarray()[0]
            ni   = rng.randint(0, len(self.train_df) - 1)
            while ni == i: ni = rng.randint(0, len(self.train_df) - 1)
            an_v = self.tfidf.vec.transform([self.train_df.iloc[ni]['Answer']]).toarray()[0]
            X.append(np.abs(q_v - ap_v)); y.append(1)
            X.append(np.abs(q_v - an_v)); y.append(0)
        print(f'[RF] training on {len(X)} pairs...')
        self.rf = RandomForestClassifier(n_estimators=100, max_depth=10,
                                         random_state=SEED, n_jobs=-1)
        self.rf.fit(X, y)
        with open(RF_MODEL_PATH, 'wb') as f:
            pickle.dump(self.rf, f)
        print('[RF] trained and saved')

    def answer(self, question, profile=None):
        q_v  = self.tfidf.vec.transform([question]).toarray()[0]
        sims = self.tfidf._cs(self.tfidf.vec.transform([question]), self.tfidf.mat).flatten()
        top  = np.argsort(sims)[::-1][:50]
        best_score, best_i = -1.0, top[0]
        for idx in top:
            a_v = self.tfidf.vec.transform([self.train_df.iloc[idx]['Answer']]).toarray()[0]
            sc  = float(self.rf.predict_proba(np.abs(q_v - a_v).reshape(1, -1))[0][1])
            if sc > best_score:
                best_score, best_i = sc, idx
        return self.train_df.iloc[best_i]['Answer']


print('all comparison model classes defined (GRU, TextCNN, TF-IDF, Random Forest)')

In [ ]:
def eval_retrieval_models(test_df, retrievers, n=N_EVAL):
    """Evaluate retrieval models on same test sample — no Gemini calls needed."""
    sample = test_df.sample(min(n, len(test_df)), random_state=SEED).reset_index(drop=True)
    results = {}
    for label, ret in retrievers.items():
        print(f'evaluating: {label}')
        r1_l, rL_l, b1_l, t_l = [], [], [], []
        for _, row in tqdm(sample.iterrows(), total=len(sample)):
            q, ref = row['Question'], row['Answer']
            t0     = time.time()
            pred   = ret.answer(q)
            ms     = (time.time() - t0) * 1000.0
            r      = get_rouge(pred, ref)
            b      = get_bleu(pred, ref)
            r1_l.append(r['rouge1']); rL_l.append(r['rougeL'])
            b1_l.append(b['bleu1']); t_l.append(ms)
        results[label] = {
            'ROUGE-1':       round(float(np.mean(r1_l)), 4),
            'ROUGE-L':       round(float(np.mean(rL_l)), 4),
            'BLEU-1':        round(float(np.mean(b1_l)), 4),
            'Avg Time (ms)': round(float(np.mean(t_l)),  2),
        }
    return results


def print_model_comparison(model_results):
    best = max(model_results, key=lambda m: model_results[m]['ROUGE-L'])
    print(f"{'Model':<28} {'ROUGE-1':>8} {'ROUGE-L':>8} {'BLEU-1':>8} {'Time ms':>10}")
    print('-' * 68)
    for m, s in model_results.items():
        tag = '  <-- BEST' if m == best else ''
        print(f"{m:<28} {s['ROUGE-1']:>8.4f} {s['ROUGE-L']:>8.4f} {s['BLEU-1']:>8.4f} {s['Avg Time (ms)']:>10.1f}{tag}")


def plot_model_comparison_bars(model_results):
    models  = list(model_results.keys())
    metrics = ['ROUGE-1', 'ROUGE-L', 'BLEU-1']
    pal3    = ['#4C72B0', '#DD8452', '#55A868']
    x = np.arange(len(models))
    w = 0.25

    fig, ax = plt.subplots(figsize=(13, 6))
    for i, (metric, col) in enumerate(zip(metrics, pal3)):
        vals = [model_results[m][metric] for m in models]
        bars = ax.bar(x + (i - 1) * w, vals, w, label=metric,
                      color=col, edgecolor='black', linewidth=0.8, alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.003,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=8)

    bi_idx = next(i for i, m in enumerate(models) if 'BiLSTM' in m)
    ymax   = max(model_results[m][met] for m in models for met in metrics)
    ax.axvspan(bi_idx - 0.44, bi_idx + 0.44, alpha=0.07, color='green', zorder=0)
    ax.text(bi_idx, ymax * 1.18, '★ Proposed\n  Model',
            ha='center', va='bottom', fontsize=9, color='darkgreen', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=10, rotation=12, ha='right')
    ax.set_ylabel('Score')
    ax.set_title('Retrieval Model Comparison — ROUGE & BLEU Scores\n'
                 'Demonstrates BiLSTM Superiority for Medical Q&A', fontweight='bold')
    ax.set_ylim(0, ymax * 1.38)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.25, linestyle='--')
    fig.tight_layout()
    save_fig(fig, '8_model_comparison_bars.png')
    plt.show()


def plot_model_response_time(model_results):
    models   = list(model_results.keys())
    times    = [model_results[m]['Avg Time (ms)'] for m in models]
    bar_cols = ['#55A868' if 'BiLSTM' in m else '#4C72B0' for m in models]

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.barh(models, times, color=bar_cols, edgecolor='black', linewidth=0.8, alpha=0.88)
    for bar, t in zip(bars, times):
        ax.text(bar.get_width() + max(times) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{t:.1f} ms', va='center', ha='left', fontsize=11, fontweight='bold')
    ax.set_xlabel('Average Response Time (ms)')
    ax.set_title('Inference Speed — All 5 Retrieval Models', fontweight='bold')
    ax.set_xlim(0, max(times) * 1.3)
    ax.grid(axis='x', alpha=0.25, linestyle='--')
    fig.tight_layout()
    save_fig(fig, '9_model_response_time.png')
    plt.show()


def plot_model_ranking(model_results):
    models = list(model_results.keys())
    scores = {
        m: round((model_results[m]['ROUGE-1'] +
                  model_results[m]['ROUGE-L'] +
                  model_results[m]['BLEU-1']) / 3.0, 4)
        for m in models
    }
    order    = sorted(models, key=lambda m: scores[m])
    vals     = [scores[m] for m in order]
    bar_cols = ['#55A868' if 'BiLSTM' in m else '#4C72B0' for m in order]

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.barh(order, vals, color=bar_cols, edgecolor='black', linewidth=0.8, alpha=0.88)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_width() + max(vals) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{v:.4f}', va='center', ha='left', fontsize=11, fontweight='bold')
    for i, m in enumerate(order):
        if 'BiLSTM' in m:
            ax.text(0.001, i, ' ← Proposed',
                    va='center', ha='left', fontsize=9, color='darkgreen', fontstyle='italic')
    ax.set_xlabel('Average Score  (ROUGE-1 + ROUGE-L + BLEU-1) / 3')
    ax.set_title('Overall Model Ranking\nProves BiLSTM is the Best Retrieval Backbone',
                 fontweight='bold')
    ax.set_xlim(0, max(vals) * 1.3)
    ax.grid(axis='x', alpha=0.25, linestyle='--')
    fig.tight_layout()
    save_fig(fig, '10_model_ranking.png')
    plt.show()


print('model comparison evaluation and figure functions defined')

In [ ]:
# ── 1. Train / load GRU model ─────────────────────────────────────────────
if GRU_MODEL_PATH.exists() and GRU_EMBS_PATH.exists():
    print('\n[GRU] Loading cached model...')
    gru_model = SiameseGRU(vocab_size=len(vocb)).to(DEVICE)
    gru_model.load_state_dict(
        torch.load(GRU_MODEL_PATH, map_location=DEVICE, weights_only=True)
    )
    gru_model.eval()
else:
    gru_model = train_comparison_model(
        SiameseGRU, len(vocb), train_df, val_df, vocb,
        GRU_MODEL_PATH, 'GRU',
    )
gru_ret = GenericNeuralRetriever(gru_model, vocb, train_df, GRU_EMBS_PATH, 'GRU')

# ── 2. Train / load TextCNN model ─────────────────────────────────────────
if CNN_MODEL_PATH.exists() and CNN_EMBS_PATH.exists():
    print('\n[TextCNN] Loading cached model...')
    cnn_model = SiameseCNN(vocab_size=len(vocb)).to(DEVICE)
    cnn_model.load_state_dict(
        torch.load(CNN_MODEL_PATH, map_location=DEVICE, weights_only=True)
    )
    cnn_model.eval()
else:
    cnn_model = train_comparison_model(
        SiameseCNN, len(vocb), train_df, val_df, vocb,
        CNN_MODEL_PATH, 'TextCNN',
    )
cnn_ret = GenericNeuralRetriever(cnn_model, vocb, train_df, CNN_EMBS_PATH, 'TextCNN')

# ── 3. TF-IDF retriever ───────────────────────────────────────────────────
tfidf_ret = TFIDFRetriever(train_df)

# ── 4. Random Forest retriever ────────────────────────────────────────────
rf_ret = RandomForestRetriever(train_df, tfidf_ret)

# ── 5. BiLSTM wrapper ─────────────────────────────────────────────────────
class _BiLSTMWrap:
    def answer(self, q, profile=None):
        return retreiver.retrieve_top1(q)['answer']

retrievers = {
    'BiLSTM (Proposed)': _BiLSTMWrap(),
    'GRU':               gru_ret,
    'TextCNN':           cnn_ret,
    'TF-IDF':            tfidf_ret,
    'Random Forest':     rf_ret,
}

# ── Evaluate (no Gemini API calls needed here) ────────────────────────────
print('\nEvaluating all 5 models on', N_EVAL, 'test questions...')
model_results = eval_retrieval_models(test_df, retrievers, n=N_EVAL)

print('\n===== ML/DL MODEL COMPARISON RESULTS =====')
print_model_comparison(model_results)

# ── Figures ───────────────────────────────────────────────────────────────
plot_model_comparison_bars(model_results)
plot_model_response_time(model_results)
plot_model_ranking(model_results)

# ── Save JSON ─────────────────────────────────────────────────────────────
(OUT_DIR / 'model_comparison.json').write_text(
    json.dumps(model_results, indent=2), encoding='utf-8'
)
print('\nmodel_comparison.json saved to', OUT_DIR)